# Sentiment-scaled CAPM — Doukas & Han (2021)

A step-by-step replication of *"Sentiment-scaled CAPM and market mispricing"*,
John A. Doukas & Xiao Han, **European Financial Management** 2021, 27:208–243.

This notebook is a teaching companion to `sentiment_capm.py`. Every code cell
below is **the exact code from that module**, in dependency order, with an
explanation of what it does and why. Running the notebook top to bottom gives
you a complete working environment — identical to `%run sentiment_capm.py`,
but with the reasoning visible.

---

## What the paper argues

The static CAPM fails empirically: market betas are not significantly priced,
and the security market line comes out flat. Doukas & Han's explanation is that
beta isn't constant — it moves with **investor sentiment**. Their fix is a
*conditional* CAPM in which the stochastic discount factor's coefficients are
allowed to depend on lagged sentiment.

Starting from a conditional SDF $M_{t+1} = a(z_t) - b(z_t)R^{em}_{t+1}$ with
sentiment $s_t$ as the conditioning instrument $z_t$, and expanding linearly
(Cochrane 2006), they arrive at their **Equation 9** (paper p. 215):

$$E(R_{i,t+1}) = r_f + \beta_{i,s}\lambda_s + \beta_{i,m}\lambda_m + \beta^s_{i,m}\lambda^s_m$$

> "The factors in this model are, the lagged sentiment, the current-period
> market return and lagged sentiment times the current-period market return."

Note carefully: **only sentiment is lagged.** The market excess return is
contemporaneous with the return it prices. That's what makes it a conditional
CAPM rather than a return-forecasting regression.

## The three specifications in this notebook

| Spec | Factors | Driver | Paper |
|---|---|---|---|
| Scaled CAPM (Eq. 9) | $s_{t-1}$, $MktRF_t$, $s_{t-1}\times MktRF_t$ | `run()` | Table 3 |
| Scaled FF3 | $s_{t-1}MktRF_t$, $s_{t-1}SMB_t$, $s_{t-1}HML_t$ | `run_ff3()` | Table 11 |
| Plain FF3 (baseline) | $MktRF_t$, $SMB_t$, $HML_t$ | `run_ff3_plain()` | Table 3, row 2 |

Plus the predictive regression (Table 2), the PLS robustness check (Table 13),
the anomaly portfolios (Tables 8/9), and the state-beta security market line
(Tables 5–7).

## Estimation method: two-pass Fama-MacBeth

**First stage** — one time-series regression per test portfolio $i$, full sample,
giving that portfolio's factor loadings:

$$R_{i,t} - r_{f,t} = a_i + b_{s,i}s_{t-1} + b_{m,i}MktRF_t + b_{sm,i}(s_{t-1}MktRF_t) + e_{i,t}$$

**Second stage** — one cross-sectional regression per month $t$, of that month's
$N$ returns on the $N$ estimated betas, giving a time series of $\lambda_t$:

$$R_{i,t} - r_{f,t} = c_t + \lambda_{s,t}b_{s,i} + \lambda_{m,t}b_{m,i} + \lambda_{sm,t}b_{sm,i} + \alpha_{i,t}$$

The reported risk premia are the time-series averages $\bar\lambda$, with
standard errors from the sample standard deviation of the monthly $\lambda_t$'s.

The paper writes out only the second-stage pricing equation; the first stage is
described in words in its Table 3 note ("The βs are estimated in the first-stage
time-series regressions on different factors"). The first-stage equation above
is the standard FM reconstruction implied by it.

---
## 0. Setup

`DATA_DIR = "00_Data"` is a **relative** path, so the working directory must be
the repository root — the folder that *contains* `00_Data`, not the folder this
notebook lives in. Adjust the path below to match your machine.

In [ ]:
import os

# must be the repo root -- the folder containing 00_Data/
os.chdir("/Users/mandarphatak/Documents/GitHub/Asset_Pricing")
print("working directory:", os.getcwd())
print("00_Data present:", os.path.isdir("00_Data"))

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.decomposition import PCA

DATA_DIR = "00_Data"

---
## 1. Test assets

The cross-sectional tests need a set of portfolios whose average returns the
model is supposed to explain. Three sets are available here, and **the choice
matters enormously** — more than any other decision in this replication.

**25 Size-BM portfolios** — the paper's own test assets, and the field standard.
Formed by independently sorting on market equity and book-to-market into 5×5 bins.

**48 Industry portfolios** — a much harder test. Lewellen, Nagel & Shanken (2010)
recommend adding industries precisely *because* they break models that look fine
on size/BM sorts. The 25 have a strong low-dimensional covariance structure (two
or three principal components explain nearly everything), so almost any three
factors correlated with those components produce a high cross-sectional $R^2$ —
LNS showed this holds even for factors built to be economically meaningless.
Industries have no such shortcut.

**32 Size-OP-INV portfolios** — a secondary robustness set.

All are Ken French value-weighted monthly returns in percent.

In [ ]:
def load_portfolios(path=f"{DATA_DIR}/clean_25_Portfolios_5x5.csv"):
    df = pd.read_csv(path, parse_dates=["Date"])
    df["ym"] = df["Date"].dt.year * 100 + df["Date"].dt.month
    df = df.drop(columns=["Date"]).set_index("ym")
    return df  # 25 columns, monthly returns in percent


def load_portfolios_32(path=f"{DATA_DIR}/clean_32_Portfolios_OP_INV.csv"):
    """Ken French 32 portfolios formed on Size x Operating Profitability x
    Investment (2x4x4), value-weighted returns. Alternative test-asset set
    to the 25 Size-BM portfolios, for a robustness check (recovered from the
    user's own ff32_panel.csv, long-format VW/EW returns since Jul 1963)."""
    df = pd.read_csv(path, parse_dates=["Date"])
    df["ym"] = df["Date"].dt.year * 100 + df["Date"].dt.month
    df = df.drop(columns=["Date"]).set_index("ym")
    return df  # 32 columns, monthly returns in percent


def load_portfolios_industry48(path=f"{DATA_DIR}/clean_48_Industry_Portfolios.csv"):
    """Ken French 48 Industry Portfolios, value-weighted monthly returns
    (percent), Jul 1926 - present. Missing-data codes (-99.99/-999) are
    already converted to NaN in the cleaned file (some industries, e.g.
    semiconductors, didn't exist yet in the early decades -- all 48 are
    fully populated from Jul 1969 onward). Alternative test-asset set to
    the 25 Size-BM / 32 Size-OP-INV portfolios."""
    df = pd.read_csv(path, parse_dates=["Date"])
    df["ym"] = df["Date"].dt.year * 100 + df["Date"].dt.month
    df = df.drop(columns=["Date"]).set_index("ym")
    return df  # 48 columns, monthly returns in percent (NaN where no firms)


_PORT_LOADERS = {"25": load_portfolios, "32": load_portfolios_32,
                  "industry48": load_portfolios_industry48}

`_PORT_LOADERS` maps a short key to a loader function. Note the key is
`"industry48"`, not `"48"` — passing `"48"` raises `KeyError`.

⚠️ **Jupyter gotcha:** this dict stores *references to function objects*, captured
at the moment the dict is built. If you later redefine `load_portfolios()` in
another cell, the dict still points at the **old** function. You must re-run the
`_PORT_LOADERS = {...}` line for the change to take effect. The same applies to
`_LOADERS` further down. This is a common source of "I changed the code but the
output didn't change".

In [ ]:
# quick look at the test assets
ports = load_portfolios()
print("25 Size-BM :", ports.shape, "|", ports.index.min(), "-", ports.index.max())
print(ports.iloc[:3, :5])

---
## 2. Fama-French factors

`Mkt-RF`, `SMB`, `HML` and `RF`, monthly, in percent.

`Mkt-RF` is **already an excess return** — the market return net of the risk-free
rate. The paper's prose alternates between calling it "the market return" and
"the market excess return", but both phrases refer to this one series; their
equations use the single symbol $MktRF_t$ throughout.

`RF` is kept because the test-asset returns arrive as *total* returns and must be
converted to excess returns by subtracting it.

In [ ]:
def load_factors(path=f"{DATA_DIR}/clean_F-F_Research_Data_Factors.csv"):
    df = pd.read_csv(path, parse_dates=["Date"])
    df["ym"] = df["Date"].dt.year * 100 + df["Date"].dt.month
    return df.drop(columns=["Date"]).set_index("ym")  # Mkt-RF, SMB, HML, RF

---
## 3. Sentiment indices

The paper uses four: **BW**, **MCSI**, **CBCCI**, and **AS** (a composite of the
first three). This replication adds **PMI** and **CFNAI** as extra robustness
checks, and **PLS** for the Table 13 analogue.

One naming confusion worth knowing: the paper's body text (p. 215) writes
"MCSI hereafter" for the Michigan index, but its Table 3 row label and
abbreviation footnote both read "MSCI". The paper is internally inconsistent.
This code uses `"mcsi"`. (MSCI, the index provider, is unrelated.)

### 3.1 Michigan Consumer Sentiment Index (MCSI)

Survey-based measure of household optimism. It is **quarterly before 1978 and
monthly after**, so the pre-1978 portion is linearly interpolated to monthly —
exactly as the paper describes in its Section 3.1.

The interpolation is done on a true datetime axis rather than on the integer
`yyyymm` code, because interpolating on the raw integer would treat the gap from
`197012` to `197101` as 89 units instead of one month.

In [ ]:
def load_mcsi(path=f"{DATA_DIR}/mcsi_raw.csv"):
    """University of Michigan ICS: quarterly through 1977, monthly from 1978.
    Linearly interpolate to monthly, as the paper does (Section 3.1)."""
    raw = pd.read_csv(path)
    month_num = {m: i for i, m in enumerate(
        ["January", "February", "March", "April", "May", "June", "July",
         "August", "September", "October", "November", "December"], start=1)}
    raw["ym"] = raw["YYYY"] * 100 + raw["Month"].map(month_num)
    raw = raw.sort_values("ym").set_index("ym")["ICS_ALL"]
    full_index = range(raw.index.min(), raw.index.max() + 1)
    full_index = [ym for ym in range(raw.index.min() // 100 * 100 + 1, raw.index.max() + 1)
                  if 1 <= ym % 100 <= 12]
    s = raw.reindex(full_index)
    # interpolate on a true time axis (months), not the coded integer ym
    dt_index = pd.to_datetime(pd.Series(full_index).astype(str), format="%Y%m")
    s.index = dt_index
    s = s.interpolate(method="linear").bfill().ffill()
    s.index = full_index
    return s.rename("sentiment")

### 3.2 PMI, Baker-Wurgler, Conference Board, CFNAI

**BW** (Baker & Wurgler 2006) is the canonical academic sentiment index and the
paper's primary measure: the first principal component of six proxies (closed-end
fund discount, IPO count, first-day IPO returns, equity share in new issues,
dividend premium, NYSE turnover), each orthogonalized to macro variables first.
We use the maintained `SENT_ORTH` series.

**CBCCI** (Conference Board Consumer Confidence) and **PMI** (ISM Manufacturing)
come from investing.com exports. Both files needed repair — see §4 below.

**CFNAI** (Chicago Fed National Activity Index) is *not* a sentiment measure at
all; it's a business-cycle activity gauge, included only as a contrast case.

In [ ]:
def load_pmi(path=f"{DATA_DIR}/pmi_ism_raw.csv"):
    df = pd.read_csv(path).sort_values("ym").set_index("ym")["pmi"]
    return df.rename("sentiment")


def load_bw(path=f"{DATA_DIR}/bw_raw.csv"):
    """Baker & Wurgler sentiment index, orthogonalized version (SENT_ORTH),
    from the officially maintained update (github.com/BWInvestorSentimentIndex).
    This is the exact series the paper uses -- monthly since July 1965,
    already orthogonalized to macro variables, no interpolation needed."""
    df = pd.read_csv(path).rename(columns={"yearmo": "ym"}).sort_values("ym").set_index("ym")
    return df["SENT_ORTH"].dropna().rename("sentiment")


def load_cbcci(path=f"{DATA_DIR}/cbcci_raw.csv"):
    """Conference Board Consumer Confidence Index (investing.com export,
    same [Date, Time, Actual, Forecast, Previous] layout as the PMI file)."""
    df = pd.read_csv(path).sort_values("ym").set_index("ym")["cbcci"]
    return df.rename("sentiment")


def load_cfnai(path=f"{DATA_DIR}/cfnai_raw.csv"):
    """Chicago Fed National Activity Index (chicagofed.org export, main
    CFNAI series). NOT one of the paper's four indices -- a business-cycle
    activity gauge, not an investor-sentiment survey. Included as an extra
    robustness check the user asked for."""
    df = pd.read_csv(path).sort_values("ym").set_index("ym")["cfnai"]
    return df.rename("sentiment")

### 3.3 Augmented Sentiment (AS) — a real PCA on our own sample

The paper's AS index is the first principal component of BW, MCSI and CBCCI, and
they report loadings of 0.318 / 0.443 / 0.452.

**Those loadings are not reusable.** They were estimated on the paper's own
1965–2015 sample. Applying them verbatim to a different sample period would be
importing someone else's estimate rather than estimating the model. So this
function runs an actual PCA on *our* standardized data.

Two details: the inputs are z-scored first (PCA on raw levels would let the
largest-variance series dominate purely through units), and the sign of a
principal component is arbitrary, so PC1 is flipped when needed to make the
loadings positive — otherwise "high sentiment" could silently mean low sentiment.

In [ ]:
def load_as(verbose=True):
    """Augmented Sentiment (AS) index -- first principal component of
    BW, MCSI and CBCCI, via sklearn's PCA on OUR OWN standardized sample
    (NOT the paper's fixed 0.318/0.443/0.452 weights, which were fit on
    their own different sample period, so reusing them verbatim would
    be wrong)."""
    bw = load_bw()
    mcsi = load_mcsi()
    cbcci = load_cbcci()
    df = pd.DataFrame({"bw": bw, "mcsi": mcsi, "cbcci": cbcci}).dropna()
    z = (df - df.mean()) / df.std()

    pca = PCA(n_components=3)
    scores = pca.fit_transform(z.values)  # T x 3, uncorrelated components
    loadings_all = pd.DataFrame(
        pca.components_.T, index=["bw", "mcsi", "cbcci"],
        columns=["PC1", "PC2", "PC3"]
    )

    pc1_loadings = loadings_all["PC1"].copy()
    if pc1_loadings.sum() < 0:  # PCA sign is arbitrary -- flip so loadings are positive
        pc1_loadings *= -1
        scores[:, 0] *= -1

    if verbose:
        print("AS index -- PCA loadings (all 3 components):")
        print(loadings_all.round(4))
        print("Explained variance ratio:", pca.explained_variance_ratio_.round(4))
        print(f"PC1 loadings used for AS:\n{pc1_loadings.round(4)}")
        print("(paper's reported loadings, for comparison: BW=0.318, MCSI=0.443, CBCCI=0.452)")

    as_idx = pd.Series(scores[:, 0], index=z.index, name="sentiment")
    return as_idx

### 3.4 PLS sentiment and the macro control variables

**PLS** (Huang, Jiang, Tu & Zhou 2015) is the Table 13 robustness measure —
sentiment extracted by partial least squares rather than principal components.
The paper doesn't construct it; it re-uses the published series.

**Controls** come from Amit Goyal's updated Welch-Goyal predictor dataset. The
paper's footnote 21 specifies the real interest rate, term premium, default
premium, inflation, and CAY. Two deviations here, both unavoidable: the term
premium uses `lty - tbl` (~20yr minus ~3mo) because Goyal's file has no 1-year
T-bill, and **CAY is omitted** because it isn't in that file. Note the paper's
own equation writes $\sum_{i=1}^{4}$ while its prose lists five controls — an
inconsistency in the paper itself.

In [ ]:
def load_pls(path=f"{DATA_DIR}/pls_sentiment_raw.csv"):
    """PLS-based sentiment index (Huang, Jiang, Tu & Zhou, 2015, RFS) --
    the orthogonalized version, from the officially maintained update
    (Fuwei Jiang's website, through Dec 2023). This is the paper's Table 13
    robustness sentiment measure -- an externally-sourced index (built via
    partial least squares instead of principal components), not something
    Doukas & Han construct themselves; they just re-use the published
    series (their footnote 26)."""
    df = pd.read_csv(path)
    s = df.set_index("yyyymm")["PLS_SENT_ORTH"]
    s.index.name = "ym"
    return s.rename("sentiment")


def load_goyal_controls(path=f"{DATA_DIR}/goyal_controls_raw.csv"):
    """Predictive-regression control variables from Amit Goyal's updated
    Welch & Goyal (2008) predictor dataset: real interest rate, term
    premium, default premium, inflation.

    Paper's exact definitions (Doukas & Han 2021, footnote 21): real rate
    = 30-day T-bill return minus inflation (matches here); default
    premium = BAA - AAA (matches here); term premium = 20-year T-bill
    MINUS 1-year T-bill (NOT matched here -- Goyal's file has no 1-year
    T-bill series, only tbl [~3-month] and lty [~20-year govt bond yield],
    so this uses the standard lty - tbl term spread as an approximation).
    CAY (Lettau-Ludvigson consumption-wealth ratio) is the paper's 5th
    control (their own equation notation says Sum_{i=1}^{4} despite the
    prose listing 5 -- an inconsistency in the paper itself); CAY isn't in
    Goyal's file and is omitted here."""
    df = pd.read_csv(path)
    df = df.set_index("yyyymm")
    df.index.name = "ym"
    return df  # real_rate, term_premium, default_premium, inflation

In [ ]:
_LOADERS = {"mcsi": load_mcsi, "pmi": load_pmi, "bw": load_bw,
            "cbcci": load_cbcci, "cfnai": load_cfnai, "as": load_as, "pls": load_pls}

Same function-reference caveat as `_PORT_LOADERS`: if you redefine any loader
above, re-run the `_LOADERS = {...}` cell or the change won't propagate.

---
## 4. The common sample window

Each series has its own coverage. If every model silently uses its own maximal
available sample, the results aren't comparable — differences between indices
get confounded with differences in sample period.

Two of the raw files also arrived **broken**, in a way that only became visible
once a common window was imposed:

- **CBCCI** appeared to be missing Jan 2008 – Aug 2014, about 80 months. The cause
  was a date-format change on investing.com: older rows are stamped on the 1st of
  the *following* month with a `(Month)` annotation giving the true reference
  month, newer rows are stamped within the month itself. Parsing the release date
  naively mangles the older block. One further row, `Nov 01, 2012`, lost its
  `(Oct)` annotation entirely and had to be reassigned to October by hand.
- **PMI** had the same format transition and a one-month hole at the seam.

After repair, both cover Dec 1969 – Aug 2026 gap-free, with a single interpolated
month each (CBCCI Jan 2008, PMI Feb 2008) at the format seam.

This is worth remembering as a methodological point: **the apparent strength of
CBCCI in early runs was an artifact of a data gap that happened to exclude the
2008 financial crisis.** Restoring those months roughly halved its sentiment
coefficient. A uniform window didn't just make results comparable — it surfaced
a bug that would otherwise have gone into the write-up as a finding.

`COMMON_WINDOW` is Dec 1969 – Dec 2025. The start is bound by CBCCI/PMI, the end
by BW's maintained update. It costs the ~8 months of 2026 that other series have.

In [ ]:
COMMON_WINDOW = (196912, 202512)


def _apply_date_range(panel, date_range):
    if date_range is None:
        return panel
    start_ym, end_ym = date_range
    return panel.loc[(panel.index >= start_ym) & (panel.index <= end_ym)]

---
## 5. Building the estimation panel

`build_panel()` joins test assets, factors and sentiment on the month index,
applies the date window, and constructs the three Eq. 9 regressors.

The order of operations matters in two places:

**Drop missing portfolio months *before* standardizing sentiment.** Some industry
portfolios have no constituent firms in early decades. If sentiment were z-scored
first, its mean and standard deviation would be computed over months that get
dropped immediately afterwards.

**Standardize sentiment *within* the window.** With `date_range` set, the z-score
uses the mean and sd of the common window, so all indices are standardized over
the same period rather than over their own idiosyncratic histories.

The timing convention, once more, because it's the single easiest thing to get
wrong: `s_lag` is sentiment at $t-1$; `mkt_rf` is the market excess return at
$t$, **contemporaneous** with the portfolio return being explained; `s_mkt` is
their product. Only sentiment is lagged.

In [ ]:
def build_panel(sentiment_kind="mcsi", portfolio_kind="25", date_range=None):
    ports = _PORT_LOADERS[portfolio_kind]()
    facs = load_factors()
    sent = _LOADERS[sentiment_kind]()

    panel = ports.join(facs, how="inner").join(sent, how="inner")
    panel = panel.sort_index()
    panel = _apply_date_range(panel, date_range)
    port_cols = list(ports.columns)
    # drop any month with a missing test-asset return (e.g. an industry
    # portfolio with no firms yet in the early decades) BEFORE standardizing
    # sentiment, so the z-score isn't computed over a period we'll drop anyway
    panel = panel.dropna(subset=port_cols)
    # standardize sentiment over the estimation sample (mean 0, sd 1), as in the paper
    # -- when date_range is set, this mean/std is computed WITHIN the common
    # window, so different indices' z-scores are standardized over the same
    # period rather than each index's own idiosyncratic full history
    panel["sentiment_z"] = (panel["sentiment"] - panel["sentiment"].mean()) / panel["sentiment"].std()
    panel["s_lag"] = panel["sentiment_z"].shift(1)
    # Doukas & Han's Eq.9 timing (confirmed against the paper): the
    # regressors are LAGGED sentiment (s_lag), the CONTEMPORANEOUS market
    # return (mkt_rf, same period t as the portfolio return), and their
    # product (lagged sentiment x contemporaneous market return). Only
    # sentiment is lagged -- Mkt-RF/SMB/HML are not. This is what makes it
    # a genuine conditional CAPM: the asset's exposure to THIS period's
    # market move, scaled by sentiment known BEFORE the period started.
    panel["mkt_rf"] = panel["Mkt-RF"]
    panel["s_mkt"] = panel["s_lag"] * panel["mkt_rf"]
    panel = panel.dropna(subset=["s_lag", "s_mkt"])

    for c in port_cols:
        panel[c] = panel[c] - panel["RF"]  # excess returns
    return panel, port_cols

In [ ]:
# build one panel and inspect what came out
panel, port_cols = build_panel("mcsi", portfolio_kind="25", date_range=COMMON_WINDOW)
print("T =", len(panel), " N =", len(port_cols))
print("sample:", panel.index.min(), "-", panel.index.max())
print()
print(panel[["sentiment", "sentiment_z", "s_lag", "mkt_rf", "s_mkt", "RF"]].head())

---
## 6. First stage — time-series betas

One OLS regression per portfolio, over the **full sample**, of that portfolio's
excess return on the three factors. This yields each portfolio's factor loadings
$(b_{s,i}, b_{m,i}, b_{sm,i})$ — the exposures that the second stage will then
price.

Returns both the $N\times K$ beta matrix and the $T\times N$ residual matrix; the
residuals are needed later for the $\chi^2$ joint test.

Note the betas are estimated once over the whole sample, not rolled. The paper
does the same, and it's standard FM practice — though it does mean the betas use
information from the full period, including months after the returns they price.

In [ ]:
def first_stage_betas(panel, port_cols, factor_cols=("s_lag", "mkt_rf", "s_mkt")):
    betas = {}
    resid = {}
    for c in port_cols:
        X = sm.add_constant(panel[list(factor_cols)])
        fit = sm.OLS(panel[c], X).fit()
        betas[c] = fit.params[list(factor_cols)].values
        resid[c] = fit.resid.values
    beta_mat = pd.DataFrame(betas, index=list(factor_cols)).T  # N x K
    resid_mat = pd.DataFrame(resid, index=panel.index)          # T x N
    return beta_mat, resid_mat

In [ ]:
beta_mat, resid_mat = first_stage_betas(panel, port_cols)
print(beta_mat.round(4).head())

---
## 7. Second stage — Fama-MacBeth cross-sectional regressions

For each month $t$, regress that month's $N$ portfolio excess returns on the $N$
betas from stage one. The betas are fixed across months, so the regressor matrix
$X$ is built once outside the loop; only the dependent variable changes.

This produces a time series of $\lambda_t$ estimates. The reported risk premium is
their average, and the classical FM standard error is their sample standard
deviation divided by $\sqrt{T}$ — the paper's footnote 16. The elegance of the FM
approach is that this standard error automatically accounts for cross-sectional
correlation in the pricing errors, because each month's $\lambda_t$ is a single
draw and the variation *across* draws captures it.

The intercept `const` is worth watching closely. Since the left-hand side is
already an *excess* return, a correctly specified model should have an intercept
of **zero**. A large, significant constant means the model is explaining the
cross-section with a level shift rather than with its factors.

In [ ]:
def fama_macbeth(panel, port_cols, beta_mat):
    X = sm.add_constant(beta_mat.values)  # N x (K+1), same every month
    lambdas = []
    for t, row in panel[port_cols].iterrows():
        y = row.values
        fit = sm.OLS(y, X).fit()
        lambdas.append(fit.params)
    lam_df = pd.DataFrame(lambdas, index=panel.index,
                           columns=["const"] + list(beta_mat.columns))
    lam_mean = lam_df.mean()
    T = len(lam_df)
    se_fm = lam_df.std(ddof=1) / np.sqrt(T)
    t_fm = lam_mean / se_fm
    return lam_df, lam_mean, se_fm, t_fm

In [ ]:
lam_df, lam_mean, se_fm, t_fm = fama_macbeth(panel, port_cols, beta_mat)
print("mean lambdas:"); print(lam_mean.round(4))
print(); print("FM t-stats:"); print(t_fm.round(3))

---
## 8. The Shanken (1992) correction

The second stage regresses on **estimated** betas, not true ones. That's a
classic errors-in-variables problem, and it makes the classical FM standard
errors too small — overstating significance.

Shanken's correction inflates them by

$$\sqrt{1 + \lambda'\Sigma_f^{-1}\lambda}$$

where $\Sigma_f$ is the covariance matrix of the realized factors.

**The correction changes standard errors only — never the coefficients.** The
$\lambda$ point estimates are identical before and after; only the $t$-statistics
shrink. If you ever see the lambdas move when you apply it, something is wrong.

The multiplier applies to the factor lambdas, not to the intercept — which is why
`run()` scales only `["s_lag", "mkt_rf", "s_mkt"]`. The paper reports both:
uncorrected $t$ on top, Shanken-corrected underneath.

In [ ]:
def shanken_correction(lam_mean, panel, factor_cols=("s_lag", "mkt_rf", "s_mkt")):
    lam = lam_mean[list(factor_cols)].values
    Sigma_f = np.cov(panel[list(factor_cols)].values.T)
    mult = 1 + lam @ np.linalg.solve(Sigma_f, lam)
    return np.sqrt(mult)  # multiply classical FM SE (of the non-constant lambdas) by this

---
## 9. Goodness of fit

### 9.1 Cross-sectional $R^2$ (OLS)

Fitted expected return for portfolio $i$ is $\hat\lambda_0 + \beta_i'\hat\lambda$,
and the pricing error is $\alpha_i = \bar R_i - \hat R_i$. The $R^2$ compares
$\sum\alpha_i^2$ against the cross-sectional variance of average returns. The
adjusted version penalizes for $K$ factors — important when comparing models with
different numbers of factors on the same assets.

In [ ]:
def cross_sectional_fit(panel, port_cols, beta_mat, lam_mean, factor_cols=("s_lag", "mkt_rf", "s_mkt")):
    avg_ret = panel[port_cols].mean()
    fitted = lam_mean["const"] + beta_mat[list(factor_cols)].values @ lam_mean[list(factor_cols)].values
    fitted = pd.Series(fitted, index=port_cols)
    alpha = avg_ret - fitted
    ss_res = (alpha ** 2).sum()
    ss_tot = ((avg_ret - avg_ret.mean()) ** 2).sum()
    r2 = 1 - ss_res / ss_tot
    N, K = len(port_cols), len(factor_cols)
    r2_adj = 1 - (1 - r2) * (N - 1) / (N - K - 1)
    return avg_ret, fitted, alpha, r2, r2_adj

### 9.2 GLS $R^2$ — and a bug worth understanding

Lewellen, Nagel & Shanken (2010) argue the OLS cross-sectional $R^2$ is close to
meaningless on the 25 size-BM portfolios, for the reason given in §1: their tight
factor structure lets almost anything fit. Their proposed diagnostic weights the
pricing errors by $\Sigma^{-1}$:

$$R^2_{GLS} = 1 - \frac{\alpha'\Sigma^{-1}\alpha}{(\bar R - \bar R_{GLS}\iota)'\Sigma^{-1}(\bar R - \bar R_{GLS}\iota)}$$

**The subtlety that broke this code once:** the $\lambda$'s used to form $\alpha$
must themselves be estimated **by GLS**. An earlier version passed in the alphas
from the OLS/Fama-MacBeth second stage and merely weighted *those* by
$\Sigma^{-1}$. But GLS is by construction the estimator that *minimises*
$\alpha'\Sigma^{-1}\alpha$; with OLS lambdas the numerator isn't minimised, the
ratio can exceed 1, and the statistic goes negative. That produced GLS $R^2$ of
−0.20 to −0.44 on the 25 portfolios.

A correct GLS $R^2$ **can never be negative** when the model contains a constant,
because it nests the constant-only benchmark that defines the denominator. The
negative values were the tell. After the fix: MCSI 0.071, CBCCI 0.104, BW 0.564 —
all positive, and in line with the paper's 0.13 / 0.21 / 0.35.

A substantive finding fell out of that diagnostic. For the scaled CAPM the FM and
GLS lambdas differ *enormously*: for MCSI, $\lambda^s_m$ is −2.46 under FM but
**+0.43** under GLS — a sign flip; for CBCCI, −1.79 versus **+0.12**. Plain FF3's
lambdas barely move between weightings. So the scaled factor's price of risk is
highly sensitive to the weighting scheme, which is precisely the fragility the
GLS diagnostic exists to expose — and it lands on the paper's headline result.

One consequence to keep in mind when reading output: the GLS $R^2$ describes the
GLS estimator while the lambda table reports FM estimates. The two columns refer
to different estimators of the same model. That's the standard convention (LNS
do it, and so does the paper's Table 3), but it's worth stating explicitly.

In [ ]:
def gls_r2(panel, port_cols, beta_mat, factor_cols=("s_lag", "mkt_rf", "s_mkt")):
    """Lewellen, Nagel & Shanken (2010) GLS cross-sectional R^2:

        1 - (a' Sigma^-1 a) / ((Rbar - Rbar_gls*iota)' Sigma^-1 (Rbar - Rbar_gls*iota))

    CRITICAL: the lambdas used to form the pricing errors 'a' are re-estimated
    here by GLS cross-sectional regression -- they are NOT the OLS/Fama-MacBeth
    lambdas reported in the lambda table. This matters because the GLS estimator
    is by construction the one that MINIMISES a'Sigma^-1 a. Feeding this formula
    alphas built from OLS lambdas leaves the numerator un-minimised, so the ratio
    can exceed 1 and the statistic can come out NEGATIVE -- which a genuine GLS
    R^2 can never be, since the model nests the constant-only benchmark that
    defines the denominator. (An earlier version of this function did exactly
    that and produced GLS R^2 of -0.20 to -0.44 on the 25 portfolios.)

    Consequence worth remembering when reading the output: the GLS R^2 describes
    the fit of the GLS estimator, while the lambda table reports FM/OLS estimates.
    That is the standard convention (it is what LNS do, and what Doukas & Han
    report in their Table 3), but the two columns do refer to different
    estimators of the same model."""
    Sigma = panel[port_cols].cov().values
    Sigma_inv = np.linalg.pinv(Sigma)
    Rbar = panel[port_cols].mean().values
    N = len(port_cols)
    iota = np.ones(N)

    X = np.column_stack([iota, beta_mat[list(factor_cols)].values])  # N x (K+1)
    lam_gls = np.linalg.solve(X.T @ Sigma_inv @ X, X.T @ Sigma_inv @ Rbar)
    alpha = Rbar - X @ lam_gls

    Rbar_gls_mean = (iota @ Sigma_inv @ Rbar) / (iota @ Sigma_inv @ iota)
    dev = Rbar - Rbar_gls_mean * iota
    return 1 - (alpha @ Sigma_inv @ alpha) / (dev @ Sigma_inv @ dev)

### 9.3 $\chi^2$ joint test of the pricing errors

Tests $H_0:$ all pricing errors are jointly zero, via
$\alpha'\widehat{Cov}(\alpha)^{-1}\alpha \sim \chi^2(N-K)$, following the paper's
footnote 18.

**Read this together with the $R^2$, never alone.** A failure to reject is not
evidence a model works — it can equally mean the pricing errors are large but
imprecisely estimated. In this replication the industry portfolios don't reject
at 5% while the 25 reject at $p<0.001$, and that's the *opposite* of what the
$R^2$ values suggest. The reason is power: with $N=48$ and a diffuse covariance
structure the test can't discriminate; with $N=25$ and a tight one, it can.

In [ ]:
def chi2_joint_test(panel, port_cols, beta_mat, alpha, resid_mat, factor_cols=("s_lag", "mkt_rf", "s_mkt")):
    T = len(panel)
    N, K = len(port_cols), len(factor_cols)
    X = sm.add_constant(beta_mat[list(factor_cols)].values)  # N x (K+1)
    P = X @ np.linalg.pinv(X.T @ X) @ X.T
    I = np.eye(N)
    Sigma_eps = resid_mat.cov().values
    cov_alpha = (1 / T) * (I - P) @ Sigma_eps @ (I - P).T
    a = alpha.values
    stat = a @ np.linalg.pinv(cov_alpha) @ a
    from scipy import stats as sstats
    df = N - K
    pval = 1 - sstats.chi2.cdf(stat, df)
    return stat, df, pval

---
## 10. State beta — the paper's central mechanism

This is the heart of the paper's contribution, and arguably matters more than the
FM lambdas.

Conditional beta in a given sentiment state is
$B_i = b_{m,i} + b_{sm,i}\cdot E[s\,|\,\text{state}]$. States follow the
Lettau-Ludvigson convention the paper adopts: **good** = months where standardized
sentiment $\geq +1$ sd, **bad** = $\leq -1$ sd.

**State beta** $= B_i(\text{bad}) - B_i(\text{good})$.

The paper's claim is that while the *static* beta-return relation is flat (the
long-standing CAPM anomaly), the *state* beta-return relation slopes **upward** —
portfolios whose betas rise in bad-sentiment states earn higher average returns.
That would give a behavioural explanation for the value premium.

A sign relationship worth internalizing, because the output looks contradictory
otherwise: since $E[s|\text{bad}] < E[s|\text{good}]$, that bracket is negative
(around −2.8 with a 1 sd split), so **state beta is positive exactly when
$b_{sm}$ is negative**. A negative $\lambda_{s\_mkt}$ in the FM table and a
positive state-beta slope are therefore *the same statement*, not conflicting
ones.

Regressions use White (HC1) robust standard errors.

In [ ]:
def state_beta_analysis(panel, port_cols, beta_mat, threshold="1sd"):
    s = panel["sentiment_z"]
    if threshold == "1sd":
        good = s >= 1.0
        bad = s <= -1.0
    else:  # mean split
        good = s >= 0
        bad = s < 0
    s_good_mean = panel.loc[good, "sentiment_z"].mean()
    s_bad_mean = panel.loc[bad, "sentiment_z"].mean()

    b_m = beta_mat["mkt_rf"]
    b_sm = beta_mat["s_mkt"]
    beta_good = b_m + b_sm * s_good_mean
    beta_bad = b_m + b_sm * s_bad_mean
    state_beta = beta_bad - beta_good  # paper's convention: bad - good

    ret_good = panel.loc[good, port_cols].mean()
    ret_bad = panel.loc[bad, port_cols].mean()
    ret_all = panel[port_cols].mean()
    ret_market_beta = beta_mat["mkt_rf"]  # static market beta proxy (b_m only)

    def slope_reg(x, y):
        X = sm.add_constant(x.values)
        fit = sm.OLS(y.values, X).fit(cov_type="HC1")
        return fit.params[1], fit.tvalues[1], fit.rsquared

    out = {}
    out["state_beta_vs_avg_return"] = slope_reg(state_beta, ret_all)
    out["bad_beta_vs_bad_return"] = slope_reg(beta_bad, ret_bad)
    out["good_beta_vs_good_return"] = slope_reg(beta_good, ret_good)
    out["static_beta_vs_avg_return"] = slope_reg(ret_market_beta, ret_all)
    table = pd.DataFrame({
        "market_beta": b_m, "state_beta": state_beta,
        "beta_good_state": beta_good, "beta_bad_state": beta_bad,
        "ret_all": ret_all, "ret_good": ret_good, "ret_bad": ret_bad,
    })
    return out, table

---
## 11. `run()` — the Eq. 9 driver

Chains everything above: build panel → first stage → Fama-MacBeth → Shanken →
fit statistics → state beta. This is the Table 3 analogue.

Everything is returned in a dict, so you can dig into `beta_mat`, `panel`, or
`state_table` afterwards rather than only reading the printed summary.

In [ ]:
def run(sentiment_kind="mcsi", threshold="1sd", portfolio_kind="25", date_range=None):
    panel, port_cols = build_panel(sentiment_kind, portfolio_kind, date_range)
    beta_mat, resid_mat = first_stage_betas(panel, port_cols)
    lam_df, lam_mean, se_fm, t_fm = fama_macbeth(panel, port_cols, beta_mat)
    shanken_mult = shanken_correction(lam_mean, panel)
    se_shanken = se_fm.copy()
    se_shanken[["s_lag", "mkt_rf", "s_mkt"]] *= shanken_mult
    t_shanken = lam_mean / se_shanken

    avg_ret, fitted, alpha, r2, r2_adj = cross_sectional_fit(panel, port_cols, beta_mat, lam_mean)
    r2gls = gls_r2(panel, port_cols, beta_mat)
    chi2, chi2_df, chi2_p = chi2_joint_test(panel, port_cols, beta_mat, alpha, resid_mat)
    state_out, state_table = state_beta_analysis(panel, port_cols, beta_mat, threshold)

    print(f"\n{'='*78}\nSENTIMENT-SCALED CAPM  --  sentiment = {sentiment_kind.upper()}"
          f"  ({threshold} good/bad split, {portfolio_kind}-portfolio test assets)\n{'='*78}")
    print(f"Sample: {panel.index.min()} - {panel.index.max()}  (T = {len(panel)} months, N = {len(port_cols)} portfolios)\n")

    print("-- Fama-MacBeth cross-sectional regression (Table 3 analogue) --")
    tbl = pd.DataFrame({
        "lambda": lam_mean, "t_FM": t_fm, "SE_FM": se_fm,
        "t_Shanken": t_shanken, "SE_Shanken": se_shanken,
    })
    print(tbl.round(4))
    print(f"\nR^2 (unadjusted): {r2:.4f}   R^2 (adjusted): {r2_adj:.4f}   R^2 (GLS): {r2gls:.4f}")
    print(f"Chi2 joint pricing-error test: {chi2:.2f}  (df={chi2_df}, p={chi2_p:.4f})")

    print("\n-- State-beta security market line (Table 5-7 analogue) --")
    for k, (b, t, r2_) in state_out.items():
        print(f"  {k:32s}  slope={b:8.4f}   t={t:7.2f}   R2={r2_:.3f}")

    return {
        "panel": panel, "port_cols": port_cols, "beta_mat": beta_mat,
        "lambda_table": tbl, "r2": r2, "r2_adj": r2_adj, "r2_gls": r2gls,
        "chi2": (chi2, chi2_df, chi2_p), "state_out": state_out,
        "state_table": state_table,
    }

In [ ]:
res = run("mcsi", portfolio_kind="25", date_range=COMMON_WINDOW)

---
## 12. Sentiment-scaled FF3 — Table 11

A different equation from Eq. 9, not an extension of it. The factors are
$s_{t-1}MktRF_t$, $s_{t-1}SMB_t$, $s_{t-1}HML_t$ — **all three are interactions**,
and there is no bare sentiment level term and no bare market term.

That absence turns out to matter. In earlier runs CBCCI looked strong under Eq. 9
specifically because of its bare `s_lag` level term; moving to this spec erased
the advantage entirely, which located the effect in the level term rather than in
the beta-scaling mechanism the paper theorizes.

SMB and HML are contemporaneous here, same as `mkt_rf` — only sentiment is lagged.

In [ ]:
def build_panel_ff3(sentiment_kind="mcsi", portfolio_kind="25", date_range=None):
    """Panel for the paper's Table 11 spec: sentiment-scaled FF3, i.e. the
    first-stage regressors are s_{t-1}*MktRF_t, s_{t-1}*SMB_t, s_{t-1}*HML_t
    -- NOT a plain s_{t-1} level term (unlike the base scaled-CAPM Eq.9).
    Only sentiment is lagged; SMB/HML are contemporaneous, same timing as
    MktRF in build_panel() (see that function's docstring)."""
    panel, port_cols = build_panel(sentiment_kind, portfolio_kind, date_range)
    panel["s_smb"] = panel["s_lag"] * panel["SMB"]
    panel["s_hml"] = panel["s_lag"] * panel["HML"]
    return panel, port_cols


def run_ff3(sentiment_kind="mcsi", portfolio_kind="25", date_range=None):
    """Table 11 analogue: E_t(R_i,t+1) = rf + b^s_i,m*lam^s_m + b^s_i,smb*lam^s_smb
    + b^s_i,hml*lam^s_hml, on the 25 Size-BM portfolios (paper tests all four
    sentiment indices against this spec)."""
    factor_cols = ("s_mkt", "s_smb", "s_hml")
    panel, port_cols = build_panel_ff3(sentiment_kind, portfolio_kind, date_range)
    beta_mat, resid_mat = first_stage_betas(panel, port_cols, factor_cols)
    lam_df, lam_mean, se_fm, t_fm = fama_macbeth(panel, port_cols, beta_mat)
    shanken_mult = shanken_correction(lam_mean, panel, factor_cols)
    se_shanken = se_fm.copy()
    se_shanken[list(factor_cols)] *= shanken_mult
    t_shanken = lam_mean / se_shanken

    avg_ret, fitted, alpha, r2, r2_adj = cross_sectional_fit(
        panel, port_cols, beta_mat, lam_mean, factor_cols)
    r2gls = gls_r2(panel, port_cols, beta_mat, factor_cols)

    print(f"\n{'='*78}\nSENTIMENT-SCALED FF3 (Table 11 analogue)  --  sentiment = {sentiment_kind.upper()}"
          f"  ({portfolio_kind}-portfolio test assets)\n{'='*78}")
    print(f"Sample: {panel.index.min()} - {panel.index.max()}  (T = {len(panel)} months, N = {len(port_cols)} portfolios)\n")
    tbl = pd.DataFrame({
        "lambda": lam_mean, "t_FM": t_fm, "SE_FM": se_fm,
        "t_Shanken": t_shanken, "SE_Shanken": se_shanken,
    })
    print(tbl.round(4))
    print(f"\nR^2 (unadjusted): {r2:.4f}   R^2 (adjusted): {r2_adj:.4f}   R^2 (GLS): {r2gls:.4f}")

    return {
        "panel": panel, "port_cols": port_cols, "beta_mat": beta_mat,
        "lambda_table": tbl, "r2": r2, "r2_adj": r2_adj, "r2_gls": r2gls,
    }

---
## 13. Plain FF3 — the unscaled baseline

Standard Fama-MacBeth on $MktRF$, $SMB$, $HML$ with no sentiment anywhere. This
is the benchmark that makes the sentiment results interpretable: without it,
"the scaled CAPM gets $R^2$ of 0.54" is a number with nothing to compare against.

`sentiment_kind` here only pins the sample window to match the sentiment runs, so
the comparison is on identical months. Both models have $K=3$, so $\chi^2$ degrees
of freedom and the adjusted-$R^2$ penalty are identical too — a clean head-to-head.

Note this function reads the **raw** `Mkt-RF`/`SMB`/`HML` columns, deliberately
bypassing the lowercase `mkt_rf` that `build_panel()` creates. That keeps it a
genuine literature-standard baseline regardless of what happens to the scaled spec.

In [ ]:
def run_ff3_plain(sentiment_kind="mcsi", portfolio_kind="25", date_range=None):
    """The 'usual' Fama-French 3-factor Fama-MacBeth test -- NOT scaled by
    sentiment (contrast with run_ff3(), which is the sentiment-SCALED FF3,
    Table 11 analogue). This is a baseline: plain Mkt-RF/SMB/HML betas
    priced via standard two-pass Fama-MacBeth, sentiment plays no role in
    the regression itself. sentiment_kind only restricts the sample to the
    same overlap window used by the sentiment-scaled tests, so R^2/chi2
    are directly comparable across models on an apples-to-apples sample."""
    factor_cols = ("Mkt-RF", "SMB", "HML")
    panel, port_cols = build_panel(sentiment_kind, portfolio_kind, date_range)
    beta_mat, resid_mat = first_stage_betas(panel, port_cols, factor_cols)
    lam_df, lam_mean, se_fm, t_fm = fama_macbeth(panel, port_cols, beta_mat)
    shanken_mult = shanken_correction(lam_mean, panel, factor_cols)
    se_shanken = se_fm.copy()
    se_shanken[list(factor_cols)] *= shanken_mult
    t_shanken = lam_mean / se_shanken

    avg_ret, fitted, alpha, r2, r2_adj = cross_sectional_fit(
        panel, port_cols, beta_mat, lam_mean, factor_cols)
    r2gls = gls_r2(panel, port_cols, beta_mat, factor_cols)
    chi2, chi2_df, chi2_p = chi2_joint_test(panel, port_cols, beta_mat, alpha, resid_mat, factor_cols)

    print(f"\n{'='*78}\nPLAIN (unscaled) FAMA-FRENCH 3-FACTOR MODEL -- baseline"
          f"  ({portfolio_kind}-portfolio test assets, sample matched to {sentiment_kind.upper()})\n{'='*78}")
    print(f"Sample: {panel.index.min()} - {panel.index.max()}  (T = {len(panel)} months, N = {len(port_cols)} portfolios)\n")
    tbl = pd.DataFrame({
        "lambda": lam_mean, "t_FM": t_fm, "SE_FM": se_fm,
        "t_Shanken": t_shanken, "SE_Shanken": se_shanken,
    })
    print(tbl.round(4))
    print(f"\nR^2 (unadjusted): {r2:.4f}   R^2 (adjusted): {r2_adj:.4f}   R^2 (GLS): {r2gls:.4f}")
    print(f"Chi2 joint pricing-error test: {chi2:.2f}  (df={chi2_df}, p={chi2_p:.4f})")

    return {
        "panel": panel, "port_cols": port_cols, "beta_mat": beta_mat,
        "lambda_table": tbl, "r2": r2, "r2_adj": r2_adj, "r2_gls": r2gls,
        "chi2": (chi2, chi2_df, chi2_p),
    }

---
## 14. Anomaly portfolios — Tables 8/9 (partial)

**This is a partial replication and the substitutions are substantial.** The paper
uses eight Stambaugh-Yu-Yuan (2012) anomalies built from CRSP/Compustat via WRDS:
asset growth, net operating assets, net stock issues, total accruals, composite
equity issuance, investment-to-assets, return on equity, failure probability.

Without WRDS access, the four closest public analogues from Ken French are
substituted — net share issues, investment, accruals, and operating profitability
standing in for ROE. So this is **4 of 8 anomalies, with different variable
definitions**. Treat these results as indicative only.

Long/short leg directions follow the standard convention that the side theory
predicts should earn more is "long": low NI/INV/AC, but *high* OP.

In [ ]:
_ANOMALY_CODES = ["NI", "INV", "AC", "OP"]
_ANOMALY_LONG_SHORT = {
    "NI": ("Dec1", "Dec10"), "INV": ("Dec1", "Dec10"),
    "AC": ("Dec1", "Dec10"), "OP": ("Dec10", "Dec1"),
}


def load_anomaly_deciles(code):
    path = f"{DATA_DIR}/clean_{code}_deciles.csv"
    df = pd.read_csv(path, parse_dates=["Date"])
    df["ym"] = df["Date"].dt.year * 100 + df["Date"].dt.month
    df = df.drop(columns=["Date"]).set_index("ym")
    df.columns = [f"{code}_{c}" for c in df.columns]
    return df  # 10 columns (Dec1..Dec10), monthly returns in percent


def load_portfolios_anomaly_pool():
    """Pooled deciles across the 4 available anomalies (40 portfolios total
    -- the paper pools 8 anomalies x 10 deciles = 80)."""
    parts = [load_anomaly_deciles(c) for c in _ANOMALY_CODES]
    df = parts[0]
    for p in parts[1:]:
        df = df.join(p, how="inner")
    return df


_PORT_LOADERS["anomaly_pool"] = load_portfolios_anomaly_pool


def _build_single_anomaly_panel(code, sentiment_kind="as"):
    ports = load_anomaly_deciles(code)
    port_cols = list(ports.columns)
    facs = load_factors()
    sent = _LOADERS[sentiment_kind]()
    panel = ports.join(facs, how="inner").join(sent, how="inner").sort_index()
    panel["sentiment_z"] = (panel["sentiment"] - panel["sentiment"].mean()) / panel["sentiment"].std()
    panel["s_lag"] = panel["sentiment_z"].shift(1)
    panel["mkt_rf"] = panel["Mkt-RF"]
    panel["s_mkt"] = panel["s_lag"] * panel["mkt_rf"]
    panel = panel.dropna(subset=["s_lag", "s_mkt"])
    for c in port_cols:
        panel[c] = panel[c] - panel["RF"]
    return panel, port_cols


def table8_state_beta(sentiment_kind="as", threshold="1sd"):
    """Table 8 Panel B analogue: state-beta regression (Tables 6/7
    methodology) run separately on each anomaly's own 10 deciles."""
    rows = {}
    for code in _ANOMALY_CODES:
        panel, port_cols = _build_single_anomaly_panel(code, sentiment_kind)
        beta_mat, _ = first_stage_betas(panel, port_cols)
        state_out, _ = state_beta_analysis(panel, port_cols, beta_mat, threshold)
        b, t, r2 = state_out["state_beta_vs_avg_return"]
        rows[code] = {"b": b, "t": t, "R2": r2}
    return pd.DataFrame(rows).T


def table9_long_short(sentiment_kind="as", threshold="1sd"):
    """Table 9 analogue: returns and conditional betas of the long and
    short legs of each anomaly, split by good/bad sentiment state."""
    rows = []
    for code in _ANOMALY_CODES:
        panel, port_cols = _build_single_anomaly_panel(code, sentiment_kind)
        long_suffix, short_suffix = _ANOMALY_LONG_SHORT[code]
        long_col, short_col = f"{code}_{long_suffix}", f"{code}_{short_suffix}"

        beta_mat, _ = first_stage_betas(panel, port_cols)
        s = panel["sentiment_z"]
        if threshold == "1sd":
            good, bad = s >= 1.0, s <= -1.0
        else:
            good, bad = s >= 0, s < 0
        s_good_mean = panel.loc[good, "sentiment_z"].mean()
        s_bad_mean = panel.loc[bad, "sentiment_z"].mean()

        def cond_beta(col):
            b_m, b_sm = beta_mat.loc[col, "mkt_rf"], beta_mat.loc[col, "s_mkt"]
            return b_m + b_sm * s_good_mean, b_m + b_sm * s_bad_mean  # (good, bad)

        long_beta_good, long_beta_bad = cond_beta(long_col)
        short_beta_good, short_beta_bad = cond_beta(short_col)

        long_ret_good, long_ret_bad = panel.loc[good, long_col].mean(), panel.loc[bad, long_col].mean()
        short_ret_good, short_ret_bad = panel.loc[good, short_col].mean(), panel.loc[bad, short_col].mean()
        ls_ret_good = long_ret_good - short_ret_good
        ls_ret_bad = long_ret_bad - short_ret_bad

        rows.append({
            "anomaly": code,
            "long_ret_good": long_ret_good, "long_ret_bad": long_ret_bad,
            "short_ret_good": short_ret_good, "short_ret_bad": short_ret_bad,
            "long_short_good": ls_ret_good, "long_short_bad": ls_ret_bad,
            "long_beta_good": long_beta_good, "long_beta_bad": long_beta_bad,
            "short_beta_good": short_beta_good, "short_beta_bad": short_beta_bad,
        })
    return pd.DataFrame(rows).set_index("anomaly")


def run_anomaly_tables(sentiment_kind="as", threshold="1sd"):
    """Driver for the partial Table 8/9 replication (see module-level note
    above on what's substituted and why)."""
    print(f"\n{'='*78}\nANOMALY PORTFOLIOS -- PARTIAL Table 8/9 analogue  --  sentiment = {sentiment_kind.upper()}"
          f"\n(4 of the paper's 8 anomalies: NI, INV, AC, OP-as-ROE -- see module docstring)\n{'='*78}")

    print("\n-- Table 8 Panel B analogue: per-anomaly state-beta regression --")
    t8 = table8_state_beta(sentiment_kind, threshold)
    print(t8.round(4))

    print("\n-- Table 8 Panel C analogue: pooled 40-portfolio FMB cross-section --")
    pooled = run(sentiment_kind, threshold=threshold, portfolio_kind="anomaly_pool")

    print("\n-- Table 9 analogue: long/short leg returns & conditional betas by state --")
    t9 = table9_long_short(sentiment_kind, threshold)
    print(t9.round(4))

    return {"table8_state_beta": t8, "pooled_fmb": pooled, "table9": t9}

---
## 15. Predictive regression — Tables 2 and 13 Panel A

A different question from everything above. Rather than explaining the
*cross-section*, this asks whether sentiment predicts the *time series* of
next-month market excess returns:

$$MktRF_{t+1} = a + b\cdot Sentiment_t + \sum\alpha_i Control_{i,t} + e_t$$

Newey-West HAC standard errors with 3 lags, since overlapping monthly return data
is serially correlated.

The timing is expressed by pushing the target forward (`shift(-horizon)`) rather
than lagging the predictors, but it's the same structure: predictors at $t$,
outcome at $t+1$.

CAY is omitted from the controls — see §3.4.

In [ ]:
def predictive_regression(sentiment_kind="mcsi", with_controls=True, horizon=1, date_range=None):
    """Table 2 / Table 13 Panel A analogue:
    MktRF_{t+1} = a + b*Sentiment_t + sum(alpha_i * Controls_t) + e_t
    (HAC/Newey-West SEs, 3 lags). NOTE: the paper's control set is real
    interest rate, inflation, term premium, default premium, AND CAY
    (Lettau-Ludvigson consumption-wealth ratio) -- CAY isn't in Goyal's
    predictor file and is omitted here (real_rate/term_premium/
    default_premium/inflation only). date_range=(start_ym, end_ym) restricts
    the sample -- including the control variables -- to a common window
    (e.g. COMMON_WINDOW) before standardizing sentiment, so different
    indices are compared on an identical sample."""
    facs = load_factors()
    sent = _LOADERS[sentiment_kind]()
    panel = facs.join(sent, how="inner").sort_index()
    control_cols = []
    if with_controls:
        controls = load_goyal_controls()
        panel = panel.join(controls, how="inner")
        control_cols = ["real_rate", "term_premium", "default_premium", "inflation"]
    panel = _apply_date_range(panel, date_range)
    panel["sentiment_z"] = (panel["sentiment"] - panel["sentiment"].mean()) / panel["sentiment"].std()
    panel["mkt_rf_fwd"] = panel["Mkt-RF"].shift(-horizon)

    cols = ["sentiment_z"] + control_cols
    reg_df = panel.dropna(subset=["mkt_rf_fwd"] + cols)
    X = sm.add_constant(reg_df[cols])
    y = reg_df["mkt_rf_fwd"]
    fit = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": 3})
    return fit, reg_df


def run_table13(sentiment_kind="pls", threshold="1sd", portfolio_kind="25", date_range=None):
    """Table 13 analogue: PLS-sentiment robustness check.
    Panel A: predictive regression of next-month market excess return on
    sentiment + macro controls (CAY omitted -- see predictive_regression
    docstring). Panel B: conditional CAPM scaled by sentiment (identical
    spec to Eq.9/Table 3 -- just reuses run())."""
    print(f"\n{'='*78}\nTABLE 13 analogue  --  sentiment = {sentiment_kind.upper()}\n{'='*78}")
    print("\n-- Panel A analogue: predictive regression (CAY control omitted -- not in Goyal's data) --")
    fit, reg_df = predictive_regression(sentiment_kind, with_controls=True, date_range=date_range)
    coef = fit.params["sentiment_z"]
    tstat = fit.tvalues["sentiment_z"]
    print(f"  beta_sentiment = {coef:.4f}   t (HAC) = {tstat:.3f}   (T = {len(reg_df)})")
    print(fit.summary().tables[1])

    print("\n-- Panel B analogue: conditional CAPM scaled by sentiment (Eq.9/Table 3 spec) --")
    panelB = run(sentiment_kind, threshold=threshold, portfolio_kind=portfolio_kind, date_range=date_range)

    return {"panelA_fit": fit, "panelB": panelB}


def run_table2(date_range=None):
    """Table 2 analogue: predictive regression of next-month market excess
    return on lagged sentiment, Panel A (univariate, Eq.11) and Panel B
    (with controls -- real rate/term premium/default premium/inflation;
    CAY omitted, see load_goyal_controls docstring), for all four indices."""
    print(f"\n{'='*78}\nTABLE 2 analogue -- sentiment predicts next-month market excess return\n{'='*78}")
    rows_a, rows_b = {}, {}
    for kind in ["bw", "mcsi", "cbcci", "as"]:
        fit_a, df_a = predictive_regression(kind, with_controls=False, date_range=date_range)
        rows_a[kind.upper()] = {
            "beta": fit_a.params["sentiment_z"], "t": fit_a.tvalues["sentiment_z"],
            "R2_pct": 100 * fit_a.rsquared, "T": len(df_a),
        }
        fit_b, df_b = predictive_regression(kind, with_controls=True, date_range=date_range)
        rows_b[kind.upper()] = {
            "beta": fit_b.params["sentiment_z"], "t": fit_b.tvalues["sentiment_z"],
            "R2_pct": 100 * fit_b.rsquared, "T": len(df_b),
        }
    panelA = pd.DataFrame(rows_a).T
    panelB = pd.DataFrame(rows_b).T
    print("\n-- Panel A: univariate (Eq.11), HAC/Newey-West(3) SEs --")
    print(panelA.round(4))
    print("\n-- Panel B: with controls (real rate, term premium[approx], default premium, inflation) --")
    print(panelB.round(4))
    return {"panelA": panelA, "panelB": panelB}

---
## 16. `run_all_sentiments()` — the comparison driver

Runs Eq. 9 across the five individual sentiment indices on one asset set and one
window, then prints a summary table. Defaults to `COMMON_WINDOW` because an
apples-to-apples comparison is the entire point of the function.

**AS is deliberately excluded.** It's constructed *from* BW, MCSI and CBCCI, so
including it alongside those three would double-count the same information.

In [ ]:
def run_all_sentiments(portfolio_kind="industry48", threshold="1sd", date_range=COMMON_WINDOW):
    """Convenience driver: runs the sentiment-scaled CAPM (Eq.9 spec, via
    run()) individually for the 5 sentiment indices -- MCSI, CBCCI, PMI, BW,
    CFNAI -- all on the SAME test-asset set and the SAME common date window
    (default COMMON_WINDOW = Dec 1969-Dec 2025), so the 5 results are
    directly comparable. Deliberately excludes the composite AS index this
    round (it's built FROM BW/MCSI/CBCCI, so including it alongside those
    three would be double-counting the same information)."""
    kinds = ["mcsi", "cbcci", "pmi", "bw", "cfnai"]
    results = {}
    rows = []
    for kind in kinds:
        res = run(kind, threshold=threshold, portfolio_kind=portfolio_kind, date_range=date_range)
        results[kind] = res
        lam = res["lambda_table"]
        rows.append({
            "sentiment": kind.upper(),
            "lam_s_lag": lam.loc["s_lag", "lambda"], "t_s_lag": lam.loc["s_lag", "t_Shanken"],
            "lam_mkt_rf": lam.loc["mkt_rf", "lambda"], "t_mkt_rf": lam.loc["mkt_rf", "t_Shanken"],
            "lam_s_mkt": lam.loc["s_mkt", "lambda"], "t_s_mkt": lam.loc["s_mkt", "t_Shanken"],
            "R2": res["r2"], "R2_gls": res["r2_gls"],
            "chi2_p": res["chi2"][2],
        })
    summary = pd.DataFrame(rows).set_index("sentiment")
    print(f"\n{'='*78}\nSUMMARY -- 5 sentiment-scaled CAPMs, common window, {portfolio_kind} portfolios\n{'='*78}")
    print(summary.round(4))
    return {"results": results, "summary": summary}

---
## 17. Reproducing each table

Every call below is independent — run whichever you need.

In [ ]:
# ---- Table 3 analogue: scaled CAPM, five indices, 25 size-BM portfolios
out_25 = run_all_sentiments(portfolio_kind="25")

# ---- the same on 48 industry portfolios (the hard test)
# out_ind = run_all_sentiments(portfolio_kind="industry48")

# ---- unscaled FF3 baseline, both asset sets
# run_ff3_plain("mcsi", portfolio_kind="25",         date_range=COMMON_WINDOW)
# run_ff3_plain("mcsi", portfolio_kind="industry48", date_range=COMMON_WINDOW)

# ---- Table 11 analogue: sentiment-scaled FF3
# run_ff3("bw", portfolio_kind="25", date_range=COMMON_WINDOW)

# ---- Table 2 analogue: predictive regressions
# run_table2(date_range=COMMON_WINDOW)

# ---- Table 13 analogue: PLS robustness
# run_table13("pls", portfolio_kind="25", date_range=COMMON_WINDOW)

# ---- Tables 8/9 analogue: anomaly portfolios (partial -- see section 14)
# run_anomaly_tables("as")

---
## 18. Reference: paper → code

| Paper | Where | This notebook |
|---|---|---|
| Eq. 9, scaled CAPM pricing equation | p. 215 | `run()` (§11) |
| Table 3, FM regressions on 25 portfolios | p. 221 | `run()`, `run_ff3_plain()` |
| Tables 5–7, state-beta SML | — | `state_beta_analysis()` (§10) |
| Tables 8/9, anomaly portfolios | — | `run_anomaly_tables()` (§14, partial) |
| Table 11, sentiment-scaled FF3 | — | `run_ff3()` (§12) |
| Table 2, predictive regression (Eq. 11) | — | `run_table2()` (§15) |
| Table 13, PLS robustness | — | `run_table13()` (§15) |
| Shanken correction, footnote 16 | — | `shanken_correction()` (§8) |
| $\chi^2$ test, footnote 18 | — | `chi2_joint_test()` (§9.3) |
| Controls, footnote 21 | — | `load_goyal_controls()` (§3.4) |

## Known deviations from the paper

1. **Sample** — 1969:12–2025:12 here versus 1965:07–2015:09 in the paper.
2. **Market return** — Ken French `Mkt-RF`; the paper uses the CRSP value-weighted
   excess return. Close but not identical.
3. **CBCCI source** — investing.com export (repaired; see §4), not the paper's source.
4. **CAY control omitted** — not present in Goyal's dataset.
5. **Term premium** — `lty - tbl` rather than 20yr minus 1yr.
6. **Anomalies** — 4 public proxies rather than the paper's 8 WRDS-built measures.
7. **AS loadings** — re-estimated by PCA on this sample rather than the paper's
   published 0.318 / 0.443 / 0.452.

Given all of that, the replication tracks the paper closely on the 25 portfolios:
signs match throughout, CBCCI's $\lambda_m$ comes out at −0.746 against their
−0.74, MCSI's intercept at 1.08 against their 1.04, and the unadjusted and
adjusted $R^2$ line up across all four models.